# Find similar cases and generate clarification questions

This notebook produces two outputs for comparison:

1. **Baseline questions** based only on the new use case.
2. **Enriched questions** informed by the full text of relevant training cases.

Retrieval uses the summary prompt and embedding model shared with the indexing notebook.

In [1]:
from pathlib import Path
import json
import os
import re

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

from prompts import (
    SUMMARY_SYSTEM_PROMPT,
    SUMMARY_USER_TEMPLATE,
)

load_dotenv()

LLM_MODEL = os.getenv("OPENAI_LLM_MODEL", "gpt-4.1-mini")
EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is missing. Copy .env.example to .env and add your key.")

client = OpenAI()

from prompts import (
    ENRICHED_QUESTION_SYSTEM_PROMPT,
    ENRICHED_QUESTION_USER_TEMPLATE,
    QUESTION_SYSTEM_PROMPT,
    QUESTION_USER_TEMPLATE,
    RELEVANCE_SYSTEM_PROMPT,
    RELEVANCE_USER_TEMPLATE,
)

## Input

Replace the example with the new use-case text.

In [9]:
USE_CASE_TEXT = """
We want to use machine learning to improve product recommendations for customers in our online store.
""".strip()

USE_CASE_TEXT = """
Floor plans give home shoppers a quick yet effective grasp of the home layout like no other media is able to offer. They not only show the scale of each space, but also the key relationships between rooms and the flow of a home.
At Zillow, we recently embarked on a journey to generate floor plans (Fig. 1) from a series of 360 degree panoramas densely captured in a home. Having our floor plan generator process automatically detect wall features such as windows, doors, and openings (WDOs) from the panoramas was an essential step. Home shoppers use WDOs to orient themselves when navigating through the home. Additionally, during the generation of floor plans, WDOs serve as key intersection points for piecing together a room shape from images taken in different locations of a home.
Fig 1. Examples of Zillow generated floor plans.
The task this blog focuses on is a small step in floor plan generation from 360-degree panoramas. Given an indoor 360 pano, can we detect and localize all WDOs using bounding boxes? A precise statement on the input/output requirement of a task is the key to clearly defining a project.
The Input: a panorama of indoor environments that is leveled. By requiring a leveled panorama, we can safely assume that the north pole aligns with the y-axis of the equirectangular projection of the panoramic image.
The Output: tight bounding boxes on the left/right, but not on the top/bottom. The top-bottom boundaries are relaxed due to horizontal distortions in the equirectangular projection. We will provide more explanations in the data/annotation collection section.
The major components of our wall feature detection development pipeline are shown in Fig. 2. It follows the universal workflow of a machine learning project with some adaptations to our business problem. Unlike research projects that start with a public dataset, we examined the existing datasets on their definitions and decided to redefine the wall feature classes including the appearance and manner of annotation that can help us draw WDOs on floor plans later on.
With the new annotations defined, it was important that our annotators produce consistent annotations of good quality. We accumulated some experience in the data labeling stage. One other highlight was our choice for modeling. There are two different routes for training an object detection model and we will explain why we went with the option of training directly on the 360 panorama. Finally, we show some qualitative and quantitative results and compare model performance against human performance. Model deployment and integration of model predictions to our pipeline will be discussed in detail in a follow-up blog.
Fig 2. Major components of the current object detection pipeline. The components in the dotted rectangle will be discussed in an upcoming blog that focuses on model deployment and integration.
Having defined the task and the input and output, the next phase was data gathering and annotation, as we modeled the problem using a supervised learning approach.
In this phase, the main question we tried to answer was which classes did we want to collect and what were the class definitions?
Let’s begin by explaining the importance of class definition with a concrete example. During our research on existing datasets, two well-known public datasets ADE20K and Open Images Dataset caught our eyes since they both have the class of doors. As seen in Fig. 3., the annotations both dataset provided are on the doors (data not shown for ADE20K). However, would the same definition address our needs? The answer to this question always lie in the final business need - to show the position of WDOs on a 2D floor plan. It was clear that the position of doors should not change no matter if the door is open or closed in the panorama. What really mattered was the space underneath the door, that functionally separates two rooms, not the location of doors or the frames that are visually in an image. From this example, we learned that there is no universal definition for an object and this unique definition tailored to our problem is the key for building a custom model to address our concrete business needs.
Fig 3. Example annotation of doors in open image dataset. Door annotation is highlighted using yellow boxes. Door annotations we look for are indicated using blue boxes. Image used in figs. 3a and 3b created by Léo Ruas, subject to CC BY 2.0 license (link). Image only shown for illustrative purposes and has not been used for training or evaluation.
Another distinction from the regular object detection task is that our input data is a 360 panorama. Unlike a perspective camera that samples a limited field of view, a 360-degree camera captures the omnidirectional view of a scene. A common way of visualizing the 360 panorama is to project it onto a single planar image via equirectangular projection . This projection will introduce a varying amount of distortions across the viewing sphere (Fig. 4a) and thus pose challenges on the definition of bounding boxes. One immediate challenge we saw was that a four-point polygon bounding box is not able to capture the full object as the object is curved. We wondered, would a bounding box covering only a portion of a door impose a negative impact on how they show up in a floor plan? Not really! The requirement of leveled equirectangular projection preserves all vertical lines and thereby the left and right boundaries. Once the left and right boundaries of a door are precisely captured and then intersected with the floor plane, the location of a door on the floor plan is determined.
Deviating from the conventional definitions, here are the definitions of our proposed three classes:
It should be noted that we do not differentiate between the interior and exterior doors because both types need to be identified on a floor plan. Closet doors are also included as interior doors.
Fig. 4 Illustration on the distortion effects created due to projection and how are doors, openings, and windows defined and labeled on 360 panoramas.
One effective strategy we followed before we began labelling data, was to go through many example images and reach an agreement on how to label them. We were surprised by how much debate we had amongst ourselves on such a simple task. What came out of these discussions are concrete instructions on how to label each class with positive and negative examples for our in-house annotators. These instructions are critical for getting good annotations.
Even with that effort, it was impossible to capture all corner cases. Some examples are illustrated in Fig. 5. This ambiguity existed in almost all datasets and there were always multiple ways of labelling the same object. The key takeaway here was the need for consistency in annotation. For example, if we decided to label shower doors, then all shower doors need to be labelled throughout the dataset. As a result, we set up daily review sessions in the first week, where annotators presented cases they had doubts about, and we discussed and reached a consensus as a team on how to label these cases. Later on if annotators had questions, they would post them in the slack channel and we can quickly follow up.
Fig. 5 Corner cases for WDO annotations. (a) How do we label occluded doors? (b) Are shower doors counted as doors? (c) How do we handle objects on loop closure? (d) Do we annotate objects in a mirror?
Within two months, we have collected annotations for about 10,000 panoramas. On average, there are 6 bounding boxes in each panorama, including 2.7 doors, 2.3 windows and 1.0 openings. This unique dataset is the core asset of this project.
Fig 6. Two routes for training a model. In route 1, perspective crops with a fixed field of view (FOV), together with the annotated bounding boxes are extracted. Then an object detection model is trained on the perspective images. During inference, the bounding boxes from the crops will need to be fused to output predicted bounding boxes for the panorama. In route 2, we could directly train an object detection model on panoramas, with or without modifications on convolutions/pooling operations.
Thanks to the recent advancement in deep learning models as part of the mature machine learning field, object detection models have been turned into basic, readily available models in many of the open source deep learning frameworks and model zoo collections. However, most of the existing models such as TensorFlow object detection API and Detectron library by Facebook , are trained on perspective images. Overall we saw two routes to train the model. In the first route, we would first convert the panorama into multiple perspective crops and then perform detections on the crops followed by fusing the detections on each crop (Route 1 in Fig. 6).
Alternatively, we could choose route 2 and apply the Convolutional Neural Network (CNN) model directly on panoramas (Route 2 in Fig. 6). On that path there are also 2 sub-options.
(a) Apply off-the-shelf CNNs and “close your eyes” by treating the equirectangular projections as “flat” images.
(b) Adapt the convolutional and pooling operations to the spherical space. Spherical convolutions have been a hot research topic. Several research papers have pointed out that features learned from CNNs on flat images are different from features from 360° images and researchers have made various explorations on modifying convolutions to adapt to spherical space .
Faced with these two routes, which one should we choose?
We chose the simplest route, 2a.. The main drive for route 1 lies in removing distortions caused by the equirectangular projection and we have good knowledge that the object detection model would work well on flat images. However, the additional extraction and fusion steps would have added an extra amount of pre(post) processing to the pipeline, as well as inference time since each panorama requires inference on multiple perspective images.
As to the sub-options in route 2, we believed the networks should be tolerant to the amount of distortions for the following reasons. First, since we required a leveled panorama as input, the horizontal lines stay horizontal in equirectangular projection. On the other hand, most objects of interest exist in the less deformed regions of the panoramas and the texture of WDOs were easily identifiable by human eyes, and likely to be picked up by convolutional filters. Finally, it is worth mentioning that we had collected a good size of annotations on the 360 panoramas. This dataset allowed us to train and evaluate directly on panorama without modifications on the object detection model.
We trained a Single Shot Multibox Detector (SSD) model and Faster R-CNN model on the panorama directly. The results look promising (Fig. 7). We report the results in the next section.
Fig. 7. Qualitative results for WDO detection on 360 panorama using Faster R-CNN. Green, red and blue boxes represent windows, doors and openings, respectively. Values in the boxes indicate the confidence score of the class.
Although in this blog model evaluation appears after data collection and model training, agreeing on the business/target metric usually happens much earlier in the life-time of ML projects. While it is natural for metrics to evolve and change over time, determining evaluation metrics during the project planning stage is considered a best practice.
Localizing WDOs is a task that humans can do well easily. For those tasks, human-level performance can give us a very good estimate on the targeted error rate.
Then how do we measure human performance of this task? First we selected over 300 panoramas for all annotators to label. We treated one judge’s annotation as ground truth and measured the precision/recall for other judges to understand annotation consistency and accuracy. Guess what? Humans did not get a perfect score on this task (Fig. 8).
We calculated the average precision of each class (Table 1). Overall, doors are most consistently identified while openings was the least consistent class. Explorations on the inconsistent annotations gave us potential causes of the human-to-human disagreement (Fig. 9).
In the first example, we saw different views on what qualified as an opening among annotators. The second example showed human error during annotation. A door was mislabeled as an opening by one annotator. The last discrepancy was subtle. When three windows were next to each other, some annotators labeled them as one window while others labeled these windows separately. These examples demonstrated that despite daily rubrics, it remains a challenge for humans to follow the labeling guidelines and produce 100% consistent annotations.
Luckily, many evaluation metrics already exist for object detection tasks. We chose to use a precision-recall curve to understand the overall performance of a model at different confidence thresholds and average precision (AP) defined in the PASCAL VOC 2010 challenge as a single-number evaluation metric.
Fig. 8 Human performance on wall feature object detection. We asked three judges A, B and C to annotate the same set of panoramas and treat one judge’s annotation as ground truth and measure the precision/recall of the rest.
Fig.9 Examples of inconsistent annotations on the same panorama. A,B,C represent annotations from 3 different annotators.
We chose two models to experiment with, SSD and Faster R-CNN. At the time of writing this blog post, these two models are good representatives for one-stage and two-stage object detection models. Among all annotated panoramas, we used 70% for training, 15% for validation and 15% for testing. The precision-recall curve gives an overview of how the model does at different confidence thresholds. For a perfect model, the area under the curve would be one. As you can see above, even with human performance, it was well below one. Similar to human performance, the models performed best with door detection, followed by windows and openings (Fig. 10).
Table 1. Average precision for inter-annotator performance and model performance for each class
We used the average precision as a single metric to compare models (Table 1). Both models performed well on 360 panoramas. Specifically, the overall performance of Faster R-CNN was better than that of SSD. This is expected as the one stage model is usually lower in accuracy in exchange for faster inference time and simpler architecture. Another interesting point from the table was that the performance of Faster R-CNN on doors and windows was very close to human performance. As to openings, the gap between the model and human performance was wider, presumably because of subjective judgement on what qualifies as an opening and the model had a difficult time finding a consistent pattern.
Fig 10. Precision-recall curve of Faster R-CNN model at IoU =0.5
In this blog post, we have walked you through the first part of the wall feature detection project. Here are some key takeaways.
In the next blog, we will discuss in depth how we designed and implemented the infrastructure that would deploy and serve our models.
We would like to thank the applied science team in RMX for all the useful discussion and support for this project.
Related Articles
Subscribe to receive daily emails for the latest Zillow news and announcements, product updates and more.
""".strip()

USE_CASE_TEXT = """
We want to create a SaaS that converts custom text input to relevant SQL query
""".strip()

if not USE_CASE_TEXT:
    raise ValueError("USE_CASE_TEXT must not be empty")

## Helpers

In [10]:
def summarize_case(case_text: str) -> str:
    response = client.responses.create(
        model=LLM_MODEL,
        instructions=SUMMARY_SYSTEM_PROMPT,
        input=SUMMARY_USER_TEMPLATE.format(case_text=case_text),
    )
    return " ".join(response.output_text.strip().splitlines())


def embed_text(text: str) -> np.ndarray:
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=text)
    return np.asarray(response.data[0].embedding, dtype=float)


def parse_json_array(raw_text: str) -> list:
    """Parse an array while tolerating an accidental Markdown code fence."""
    text = raw_text.strip()
    fenced = re.fullmatch(r"```(?:json)?\s*(.*?)\s*```", text, flags=re.DOTALL | re.IGNORECASE)
    if fenced:
        text = fenced.group(1)
    value = json.loads(text)
    if not isinstance(value, list):
        raise ValueError("The model response must be a JSON array")
    return value


def ask_for_json_array(system_prompt: str, user_prompt: str) -> list:
    response = client.responses.create(
        model=LLM_MODEL,
        instructions=system_prompt,
        input=user_prompt,
    )
    return parse_json_array(response.output_text)


def cosine_similarity(query: np.ndarray, matrix: np.ndarray) -> np.ndarray:
    query_norm = np.linalg.norm(query)
    row_norms = np.linalg.norm(matrix, axis=1)
    if query_norm == 0 or np.any(row_norms == 0):
        raise ValueError("Cannot compute cosine similarity for a zero vector")
    return (matrix @ query) / (row_norms * query_norm)

## 1. Baseline questions from the input only

In [11]:
baseline_questions = ask_for_json_array(
    QUESTION_SYSTEM_PROMPT,
    QUESTION_USER_TEMPLATE.format(case_text=USE_CASE_TEXT),
)

print("Baseline questions")
for number, question in enumerate(baseline_questions, start=1):
    print(f"{number}. {question}")

Baseline questions
1. What specific business goals or success metrics do you aim to achieve with the SQL query generation feature?
2. Who are the primary users of this SaaS, and what is their level of expertise with SQL?
3. What types of databases or SQL dialects should the generated queries support?
4. Are there particular domains or industries the SQL queries need to be tailored for?
5. What is the expected complexity of the SQL queries to be generated (e.g., simple selects, joins, aggregations)?
6. Will the system need to handle ambiguous or incomplete text inputs, and if so, how should it respond?
7. What data sources or schema information will the system have access to in order to generate accurate SQL queries?
8. Do you have any baseline models or existing solutions to compare the generated queries against?
9. Are there security or privacy constraints related to user input or generated queries that must be addressed?
10. What are the expected latency and throughput requirements f

## 2. Summarize and retrieve the five nearest cases

In [12]:
INDEX_PATH = Path("data/train/train_index.csv")
TRAIN_DIR = Path("data/train/files")
TOP_K = int(os.getenv("CASE_FINDER_TOP_K", "5"))
if TOP_K < 1:
    raise ValueError("CASE_FINDER_TOP_K must be a positive integer")

if not INDEX_PATH.exists():
    raise FileNotFoundError("Build data/train/train_index.csv with prepare-train-data.ipynb first")

index_df = pd.read_csv(INDEX_PATH)
required_columns = {"id", "summary", "embedding"}
if not required_columns.issubset(index_df.columns):
    raise ValueError(f"Index must contain columns: {sorted(required_columns)}")
if index_df.empty:
    raise ValueError("The training index is empty")

case_summary = summarize_case(USE_CASE_TEXT)
query_embedding = embed_text(case_summary)
embedding_matrix = np.vstack([
    np.asarray(json.loads(value), dtype=float) for value in index_df["embedding"]
])
if embedding_matrix.shape[1] != query_embedding.size:
    raise ValueError(
        "Embedding dimensions differ. Rebuild the index with the configured embedding model."
    )

index_df = index_df.copy()
index_df["similarity"] = cosine_similarity(query_embedding, embedding_matrix)
top_candidates = index_df.nlargest(min(TOP_K, len(index_df)), "similarity").reset_index(drop=True)

print("Input summary:", case_summary)
top_candidates[["id", "similarity", "summary"]]

Input summary: The business problem involves enabling users with limited SQL expertise to interact with databases through natural language input, targeting data analysts, business users, and developers who need quick, accurate access to data without writing complex queries. The proposed solution is a SaaS platform that leverages machine learning and natural language processing models to semantically interpret custom text inputs and translate them into precise, context-aware SQL queries. Important data considerations include access to schema metadata, domain-specific terminology, and ensuring query correctness and security constraints. The expected outcome is an intuitive tool that reduces the barrier to data retrieval, accelerates insights, and minimizes errors in SQL query generation.


,id,similarity,summary
0,case_036_disdoc,0.701309,LinkedIn developed an internal AI assistant ca...
1,case_000_disdoc,0.620176,"Swiggy developed ""Hermes,"" a generative AI-pow..."
2,case_057_disdoc,0.524566,Whatnot improved its e-commerce search by addr...
3,case_037_disdoc,0.504094,"Digits developed ""Boost,"" an NLP-driven system..."
4,case_039_disdoc,0.499649,Leboncoin developed an internal chatbot assist...


## 3. Let the LLM remove false-positive candidates

Integer positions are used in the prompt so case IDs remain opaque and cannot influence the relevance judgment.

In [13]:
candidate_payload = "\n\n".join(
    f"Candidate {position}: {row.summary}"
    for position, row in enumerate(top_candidates.itertuples(index=False), start=1)
)
selected_positions_raw = ask_for_json_array(
    RELEVANCE_SYSTEM_PROMPT,
    RELEVANCE_USER_TEMPLATE.format(
        target_summary=case_summary,
        candidates=candidate_payload,
    ),
)

valid_positions = set(range(1, len(top_candidates) + 1))
if any(type(value) is not int or value not in valid_positions for value in selected_positions_raw):
    raise ValueError("The relevance response contains an invalid candidate ID")
selected_positions = list(dict.fromkeys(selected_positions_raw))
relevant_candidates = top_candidates.iloc[[position - 1 for position in selected_positions]].copy()

if relevant_candidates.empty:
    print("No candidate passed the LLM relevance filter.")
else:
    display(relevant_candidates[["id", "similarity", "summary"]])

,id,similarity,summary
0,case_036_disdoc,0.701309,LinkedIn developed an internal AI assistant ca...
1,case_000_disdoc,0.620176,"Swiggy developed ""Hermes,"" a generative AI-pow..."


## 4. Load complete reference cases and generate enriched questions

In [14]:
case_paths_by_id = {path.stem: path for path in TRAIN_DIR.glob("*.md")}
missing_case_ids = [case_id for case_id in relevant_candidates["id"] if case_id not in case_paths_by_id]
if missing_case_ids:
    raise FileNotFoundError(f"Full case files not found for IDs: {missing_case_ids}")

reference_sections = []
for case_id in relevant_candidates["id"]:
    full_text = case_paths_by_id[case_id].read_text(encoding="utf-8")
    reference_sections.append(f"<reference id={json.dumps(case_id)}>\n{full_text}\n</reference>")

reference_cases = (
    "\n\n".join(reference_sections)
    if reference_sections
    else "No relevant reference cases were found."
)
enriched_questions = ask_for_json_array(
    ENRICHED_QUESTION_SYSTEM_PROMPT,
    ENRICHED_QUESTION_USER_TEMPLATE.format(
        case_text=USE_CASE_TEXT,
        reference_cases=reference_cases,
    ),
)

print("Enriched questions")
for number, question in enumerate(enriched_questions, start=1):
    print(f"{number}. {question}")

Enriched questions
1. What are the primary user roles and user profiles expected to use the SaaS solution (e.g., data analysts, business users, product managers), and what is their expected level of SQL expertise?
2. What is the anticipated scale and complexity of the underlying databases (number of tables, schemas, size of data) that the SaaS will support for query generation?
3. Will the SaaS integrate with existing data catalogs or metadata stores to obtain schema information, table descriptions, and field attributes, or will it rely on user-provided metadata?
4. What mechanisms will be in place to handle ambiguous or incomplete user text inputs, especially when queries lack clear context or domain-specific terminology?
5. How will the system manage user-specific or organization-specific context (e.g., team, role, data access restrictions) to personalize query generation?
6. What are the requirements for data access controls and permissions enforcement to ensure generated SQL only t

## Outputs for future evaluation

Use `baseline_questions` and `enriched_questions` as the two systems to compare. `top_candidates` and `relevant_candidates` preserve retrieval diagnostics for later analysis.

In [15]:
results = {
    "input": USE_CASE_TEXT,
    "summary": case_summary,
    "baseline_questions": baseline_questions,
    "retrieved_case_ids": top_candidates["id"].tolist(),
    "relevant_case_ids": relevant_candidates["id"].tolist(),
    "enriched_questions": enriched_questions,
}
results

{'input': 'We want to create a SaaS that converts custom text input to relevant SQL query',
 'summary': 'The business problem involves enabling users with limited SQL expertise to interact with databases through natural language input, targeting data analysts, business users, and developers who need quick, accurate access to data without writing complex queries. The proposed solution is a SaaS platform that leverages machine learning and natural language processing models to semantically interpret custom text inputs and translate them into precise, context-aware SQL queries. Important data considerations include access to schema metadata, domain-specific terminology, and ensuring query correctness and security constraints. The expected outcome is an intuitive tool that reduces the barrier to data retrieval, accelerates insights, and minimizes errors in SQL query generation.',
 'baseline_questions': ['What specific business goals or success metrics do you aim to achieve with the SQL que